# Point-target tracking with colored-noise MPPI

A planar double integrator starts at (−9, −9) and tracks a stationary target at (−4, −4). Colored noise correlates acceleration perturbations across the horizon so that one candidate can suggest a sustained maneuver. [MPPI-Generic example](https://github.com/ACDSLab/MPPI-Generic/blob/abcd1ab419cb27efa113573932adbde1e0380835/examples/double_integrator_example.cu)

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415. All rights reserved.  
Authors: Frank Dellaert and Akash Ratheesh. See LICENSE for details.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam_unstable/examples/planning/DoubleIntegratorMPPI.ipynb)

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
import plotly.graph_objects as go
from IPython.display import HTML, display

## Planning horizon and running costs

The horizon spans 65 control steps of 0.015 s. The control covariance, unit spectral exponent and temperature match the upstream colored-noise example. Set `exponent = 0.0` to run the white-noise alternative. The CPU demonstration uses the upstream 128 candidates and 300 decisions.

In [3]:
horizon, samples, decisions = 65, 128, 300
control_dim, dt, exponent = 2, 0.015, 1.0
x = np.array([-9.0, -9.0, 0.1, 0.1])
goal = np.array([-4.0, -4.0, 0.0, 0.0])
coefficients = np.array([5.0, 5.0, 0.5, 0.5])
options = gtsam.GaussianProposalOptions()
options.includeNominal = True
proposal = gtsam.ColoredNoiseProposal(0.25 * np.eye(2), samples, exponent, 42, options)
rule = gtsam.MPPI(1.0)

Each state key carries a vector. A diagonal prior produces the desired running cost because GTSAM computes half the squared whitened residual. Scaling the noise sigmas by the square root of the horizon averages those costs over the horizon.

In [4]:
X = [gtsam.symbol("x", k) for k in range(horizon + 1)]
U = [gtsam.symbol("u", k) for k in range(horizon)]
graph = gtsam.NonlinearFactorGraph()
noise = gtsam.noiseModel.Diagonal.Sigmas(np.sqrt(horizon / (2 * coefficients)))
for key in X[:-1]:
    graph.add(gtsam.PriorFactorVector(key, goal, noise))
planner = gtsam.SamplingPlannerVectorVector(X, U, dt)

## Receding-horizon control

The planner uses the deterministic double-integrator vector field. The plant takes the same forward Euler step as the upstream CPU example. Each new search starts at the measured state with the previous solution shifted forward.

In [5]:
def dynamics(state, control):
    return np.array([state[2], state[3], control[0], control[1]])

def plant_step(state, control):
    return state + dt * dynamics(state, control)

In [6]:
history = [x.copy()]
nominal = np.zeros((control_dim, horizon))
for decision in range(decisions):
    initial = gtsam.Values()
    initial.insert(X[0], x)
    for k, key in enumerate(U):
        initial.insert(key, nominal[:, k].copy())
    plan = planner.optimize(graph, initial, dynamics, proposal, rule, 1)
    controls = np.column_stack([plan.atVector(key) for key in U])
    x = plant_step(x, controls[:, 0])
    history.append(x.copy())
    nominal = np.column_stack((controls[:, 1:], controls[:, -1]))
history = np.asarray(history)
assert np.isfinite(history).all()
print("Initial and final states", history[0], history[-1])

Initial and final states [-9.  -9.   0.1  0.1] [-4.52572092 -4.69772406 -0.55916741 -0.8549904 ]


## Closed-loop trajectory

The dashed line marks the target coordinates. The plotted trajectory spans 4.5 seconds. The interactive figure uses a version-pinned Plotly CDN and requires an internet connection.

In [7]:
time = dt * np.arange(len(history))
fig = go.Figure()
for axis, label in enumerate(["x", "y"]):
    fig.add_scatter(x=time, y=history[:, axis], name=f"{label} position")
fig.add_hline(y=goal[0], line_dash="dash", annotation_text="Target x and y")
fig.update_layout(xaxis_title="Time (s)", yaxis_title="Position (m)")
from html import escape

document = fig.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display(HTML(f'<iframe title="Point-target tracking with colored-noise MPPI" style="width:100%;border:0" '
             f'height="{fig.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'))

/Users/dellaert/miniforge3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead

